# Libraries

In [99]:
pip install ucimlrepo

In [100]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn import preprocessing
from sklearn.pipeline import make_pipeline
from ucimlrepo import fetch_ucirepo
import shap

# Raw Data

In [101]:
# fetch dataset
cdc_diabetes_health_indicators = fetch_ucirepo(id=891)

# data (as pandas dataframes)
X = cdc_diabetes_health_indicators.data.features
Y = cdc_diabetes_health_indicators.data.targets

# metadata
print(cdc_diabetes_health_indicators.metadata)

# variable information
print(cdc_diabetes_health_indicators.variables)

{'uci_id': 891, 'name': 'CDC Diabetes Health Indicators', 'repository_url': 'https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators', 'data_url': 'https://archive.ics.uci.edu/static/public/891/data.csv', 'abstract': 'The Diabetes Health Indicators Dataset contains healthcare statistics and lifestyle survey information about people in general along with their diagnosis of diabetes. The 35 features consist of some demographics, lab test results, and answers to survey questions for each patient. The target variable for classification is whether a patient has diabetes, is pre-diabetic, or healthy. ', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Tabular', 'Multivariate'], 'num_instances': 253680, 'num_features': 21, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Sex', 'Age', 'Education Level', 'Income'], 'target_col': ['Diabetes_binary'], 'index_col': ['ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_

In [102]:

print(cdc_diabetes_health_indicators.variables)

                    name     role     type      demographic  \
0                     ID       ID  Integer             None   
1        Diabetes_binary   Target   Binary             None   
2                 HighBP  Feature   Binary             None   
3               HighChol  Feature   Binary             None   
4              CholCheck  Feature   Binary             None   
5                    BMI  Feature  Integer             None   
6                 Smoker  Feature   Binary             None   
7                 Stroke  Feature   Binary             None   
8   HeartDiseaseorAttack  Feature   Binary             None   
9           PhysActivity  Feature   Binary             None   
10                Fruits  Feature   Binary             None   
11               Veggies  Feature   Binary             None   
12     HvyAlcoholConsump  Feature   Binary             None   
13         AnyHealthcare  Feature   Binary             None   
14           NoDocbcCost  Feature   Binary             

In [103]:
X.tail()

,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
253675,1,1,1,45,0,0,0,0,1,1,...,1,0,3,0,5,0,1,5,6,7
253676,1,1,1,18,0,0,0,0,0,0,...,1,0,4,0,0,1,0,11,2,4
253677,0,0,1,28,0,0,0,1,1,0,...,1,0,1,0,0,0,0,2,5,2
253678,1,0,1,23,0,0,0,0,1,1,...,1,0,3,0,0,0,1,7,5,1
253679,1,1,1,25,0,0,1,1,1,0,...,1,0,2,0,0,0,0,9,6,2


In [104]:
Y.tail()

,Diabetes_binary
253675,0
253676,1
253677,0
253678,0
253679,1


# Experimental Data

In [157]:
seeds = range(10)
print(np.ravel(seeds))

[0 1 2 3 4 5 6 7 8 9]


In [ ]:
res_all = pd.DataFrame()

for s in seeds:
  print("Seed: " + str(s))

  # Train-test split
  X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.1, shuffle = True, stratify = Y, random_state=s)

  # Fit scaling
  scaler = StandardScaler().fit(X_train)

  # Get predictions
  model = RandomForestClassifier(random_state=s).fit(scaler.transform(X_train), np.ravel(Y_train))
  Y_hat = model.predict(scaler.transform(X_test))

  res = X_test.copy()
  res['Y_hat'] = Y_hat
  res['Y'] = Y_test
  res['seed'] = s
  res['did'] = res.index
  #res = res.reset_index(drop=True)

  # Get SHAP values -- Tree
  random.seed(s)
  tree_explainer = shap.TreeExplainer(model,
                                    feature_perturbation="tree_path_dependent")
  shap_tree = tree_explainer.shap_values(scaler.transform(X_test), approximate=True)
  shap_tree = pd.DataFrame(shap_tree[:, :, 1], columns=[f'SHAP_TREE_{col}' for col in X_test.columns])
  shap_tree.index = X_test.index
  shap_tree['did'] = shap_tree.index

  res = res.set_index("did").join(shap_tree.set_index("did"))
  res['did'] = res.index

  # Get SHAP values -- All data points
  random.seed(s)
  tree_explainer_all = shap.TreeExplainer(model,
                                    scaler.transform(X_test.sample(n=100, random_state=s)),
                                    feature_perturbation="interventional")
  shap_all = tree_explainer_all.shap_values(scaler.transform(X_test), approximate=True)
  shap_all = pd.DataFrame(shap_all[:, :, 1], columns=[f'SHAP_ALL_{col}' for col in X_test.columns])
  shap_all.index = X_test.index
  shap_all['did'] = shap_all.index

  res = res.set_index("did").join(shap_all.set_index("did"))
  res['did'] = res.index

  # Get SHAP values -- Y
  res_Y1 = res.loc[res.Y==1, np.append(X_test.columns, 'did')]
  res_Y0 = res.loc[res.Y==0, np.append(X_test.columns, 'did')]

  random.seed(s)
  tree_explainer_Y1 = shap.TreeExplainer(model,
                                      scaler.transform(res_Y1.drop('did', axis=1).sample(n=100, random_state=s)),
                                      feature_perturbation="interventional")
  shap_Y1 = tree_explainer_Y1.shap_values(scaler.transform(res_Y1.drop('did', axis=1)), approximate=True)
  shap_Y1 = pd.DataFrame(shap_Y1[:, :, 1], columns=[f'SHAP_Y_{col}' for col in X_test.columns])
  shap_Y1.index = res_Y1.index

  random.seed(s)
  tree_explainer_Y0 = shap.TreeExplainer(model,
                                      scaler.transform(res_Y0.drop('did', axis=1).sample(n=100, random_state=s)),
                                      feature_perturbation="interventional")
  shap_Y0 = tree_explainer_Y0.shap_values(scaler.transform(res_Y0.drop('did', axis=1)), approximate=True)
  shap_Y0 = pd.DataFrame(shap_Y0[:, :, 1], columns=[f'SHAP_Y_{col}' for col in X_test.columns])
  shap_Y0.index = res_Y0.index

  shap_Y = pd.concat([shap_Y1, shap_Y0])
  shap_Y['did'] = shap_Y.index

  res = res.set_index("did").join(shap_Y.set_index("did"))
  res['did'] = res.index

  # Get SHAP values -- YHAT
  res_Y1 = res.loc[res.Y_hat==1, np.append(X_test.columns, 'did')]
  res_Y0 = res.loc[res.Y_hat==0, np.append(X_test.columns, 'did')]

  random.seed(s)
  tree_explainer_Y1 = shap.TreeExplainer(model,
                                      scaler.transform(res_Y1.drop('did', axis=1).sample(n=100, random_state=s)),
                                      feature_perturbation="interventional")
  shap_Y1 = tree_explainer_Y1.shap_values(scaler.transform(res_Y1.drop('did', axis=1)), approximate=True)
  shap_Y1 = pd.DataFrame(shap_Y1[:, :, 1], columns=[f'SHAP_YHAT_{col}' for col in X_test.columns])
  shap_Y1.index = res_Y1.index

  random.seed(s)
  tree_explainer_Y0 = shap.TreeExplainer(model,
                                      scaler.transform(res_Y0.drop('did', axis=1).sample(n=100, random_state=s)),
                                      feature_perturbation="interventional")
  shap_Y0 = tree_explainer_Y0.shap_values(scaler.transform(res_Y0.drop('did', axis=1)), approximate=True)
  shap_Y0 = pd.DataFrame(shap_Y0[:, :, 1], columns=[f'SHAP_YHAT_{col}' for col in X_test.columns])
  shap_Y0.index = res_Y0.index

  shap_Y = pd.concat([shap_Y1, shap_Y0])
  shap_Y['did'] = shap_Y.index

  res = res.set_index("did").join(shap_Y.set_index("did"))
  res['did'] = res.index

  # Concat results  
  res_all = pd.concat([res_all, res]).reset_index(drop=True)


Seed: 0
Seed: 1
Seed: 2
Seed: 3
Seed: 4
Seed: 5
Seed: 6
Seed: 7
Seed: 8
Seed: 9


In [159]:
res_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 253680 entries, 0 to 253679
Columns: 109 entries, HighBP to did
dtypes: float64(84), int64(25)
memory usage: 211.0 MB


In [160]:
res_all.head()

,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,SHAP_YHAT_NoDocbcCost,SHAP_YHAT_GenHlth,SHAP_YHAT_MentHlth,SHAP_YHAT_PhysHlth,SHAP_YHAT_DiffWalk,SHAP_YHAT_Sex,SHAP_YHAT_Age,SHAP_YHAT_Education,SHAP_YHAT_Income,did
0,1,1,1,27,0,0,0,1,0,0,...,0.000133,0.020511,0.011217,-0.009551,0.009999,0.003447,0.042725,-0.003710,-0.017349,144039
1,1,1,1,29,0,0,0,0,0,0,...,0.030133,0.004469,0.042722,0.040400,-0.009230,0.041187,0.031286,0.045541,0.045877,183177
2,0,0,0,25,0,0,0,1,1,1,...,0.000000,-0.034992,0.003408,-0.003388,-0.006562,0.003260,-0.016848,-0.003772,-0.005673,138154
3,1,1,1,43,0,0,0,0,1,1,...,0.024040,0.032908,0.009179,0.074907,-0.015563,-0.020890,-0.091034,0.042780,-0.034186,192305
4,1,1,1,36,1,0,1,1,0,1,...,0.000000,0.005917,-0.006515,-0.029036,-0.017805,-0.001998,0.023888,0.025349,0.063283,2550


In [161]:
res_all.tail()

,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,SHAP_YHAT_NoDocbcCost,SHAP_YHAT_GenHlth,SHAP_YHAT_MentHlth,SHAP_YHAT_PhysHlth,SHAP_YHAT_DiffWalk,SHAP_YHAT_Sex,SHAP_YHAT_Age,SHAP_YHAT_Education,SHAP_YHAT_Income,did
253675,0,0,1,32,0,0,0,1,0,1,...,0.000000,-0.032054,-0.008777,-0.005359,-0.011027,0.002254,-0.055150,0.010722,0.001367,187120
253676,0,1,1,23,1,0,0,0,1,1,...,0.000000,0.038701,0.008209,-0.004448,-0.010556,-0.002620,-0.032364,0.023464,-0.013789,207441
253677,0,1,1,22,1,0,1,1,1,0,...,0.000000,-0.048140,0.010026,-0.006871,-0.016610,-0.000039,0.012936,0.017764,0.031277,67224
253678,0,0,1,23,0,0,0,1,1,1,...,0.000000,-0.045495,-0.000103,-0.006943,-0.012029,-0.001597,0.020781,-0.022286,0.012165,58008
253679,1,1,1,28,0,0,0,0,1,1,...,0.002857,0.036808,0.096697,-0.012447,-0.016376,0.004312,0.020769,0.042749,0.065761,239752


In [162]:
res_all.to_csv('experimental_data.csv', index=False)